# Experiment 1 Feature Extraction from Zip Archive

Keep `cv-project-exp1-rerender-colab.zip` on Google Drive, unzip it into Colab local disk, run feature extraction locally, then copy generated feature caches back to Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Keep only the zip on Drive. The uncompressed project lives on Colab local disk.
drive.mount('/content/drive')
ZIP_PATH = Path('/content/drive/MyDrive/cv-project-exp1-rerender-colab.zip')
UNPACK_BASE = Path('/content/cv-project')
DRIVE_RESULTS = Path('/content/drive/MyDrive/cv-project-exp1-rerender-results')


def _find_exp1_root(base: Path) -> Path:
    """Zip archives often add one top-level folder; resolve to the tree that has exp1/."""
    marker = base / 'exp1' / 'features' / '__init__.py'
    if marker.is_file():
        return base
    for child in sorted(base.iterdir()):
        if child.is_dir() and (child / 'exp1' / 'features' / '__init__.py').is_file():
            return child
    raise FileNotFoundError(
        f'Could not find exp1 package under {base}. '
        'Re-zip the repo so it includes the full exp1/ directory.'
    )


assert ZIP_PATH.is_file(), f'Missing zip file: {ZIP_PATH}'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

!rm -rf /content/cv-project
!mkdir -p /content/cv-project
!unzip -q -o "{ZIP_PATH}" -d "{UNPACK_BASE}"

WORKDIR = _find_exp1_root(UNPACK_BASE)
os.environ['CV_PROJECT_ROOT'] = str(WORKDIR)
os.environ['PYTHONPATH'] = str(WORKDIR)
%cd {WORKDIR}
!nvidia-smi

In [ ]:
import sys

!{sys.executable} -m pip install -q -r requirements.txt
!{sys.executable} -m pip install -q transformers open_clip_torch pyarrow safetensors accelerate

In [ ]:
from pathlib import Path
import pandas as pd

LOCAL_PROJECT_ROOT = Path('/Users/jerry/cv-project')
PATH_COLUMNS = ['rgb_path', 'depth_path', 'normal_path', 'mask_path']

def make_colab_manifest(source: str, output: str) -> Path:
    source_path = Path(source)
    output_path = Path(output)
    df = pd.read_parquet(source_path)
    for column in PATH_COLUMNS:
        if column not in df.columns:
            continue
        def rebase(value):
            path = Path(str(value))
            if path.is_absolute():
                try:
                    path = path.relative_to(LOCAL_PROJECT_ROOT)
                except ValueError:
                    path = Path(*path.parts[1:])
            return str(path)
        df[column] = df[column].map(rebase)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(output_path, index=False)
    print(f'Wrote {len(df)} rows to {output_path}')
    return output_path

MAIN_MANIFEST = make_colab_manifest(
    'data/exp1_under12h/manifests/render_valid.parquet',
    'data/exp1_under12h/manifests/render_valid_colab.parquet',
)
DENSE_MANIFEST = make_colab_manifest(
    'data/exp1_under12h_dense/manifests/render_valid.parquet',
    'data/exp1_under12h_dense/manifests/render_valid_colab.parquet',
)

In [ ]:
# Main global CLS features: 8,307 renders x 3 models x 4 layers.
!PYTHONPATH=. python scripts/extract_exp1_features.py --config configs/exp1_under12h.yaml --render-manifest {MAIN_MANIFEST} --feature-dir data/exp1_under12h/features --models clip_vit_b16 clip_vit_l14 dinov2_vit_b --layers final layer4 layer8 layer12 --device cuda --batch-size 64 --num-workers 4 --amp

In [ ]:
# Dense patch-token features used by dense depth and dense surface-normal probes.
!PYTHONPATH=. python scripts/extract_exp1_patch_features.py --config configs/exp1_under12h_dense.yaml --render-manifest {DENSE_MANIFEST} --feature-dir data/exp1_under12h_dense/features --models clip_vit_b16 dinov2_vit_b --layers final layer8 --include-cls --dtype float16 --device cuda --batch-size 48 --num-workers 4 --amp

In [ ]:
# Copy generated feature caches back to Drive without copying the unzipped renders.
import shutil

for rel in ['data/exp1_under12h/features', 'data/exp1_under12h_dense/features']:
    src = WORKDIR / rel
    dst = DRIVE_RESULTS / rel
    if dst.exists():
        shutil.rmtree(dst)
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst)
        print(f'Copied {src} -> {dst}')
    else:
        print(f'Skipped missing {src}')